## Assignment for week 3.

Building a synthetic data generator through open source models.

The idea is to use gradio as a frontend: pick a model and a number of rows,
and get back a generated pandas dataframe.

In [ ]:
# Loading dependencies
from dotenv import load_dotenv
import ast
import os

import torch
import gradio as gr
import pandas as pd
from transformers import pipeline

In [ ]:
# Loading env
load_dotenv()
hf_key = os.getenv("HUGGING_FACE")

In [ ]:
# Writing the function first

MODELS = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'meta-llama/Meta-Llama-3.1-8B-Instruct',
]

_pipelines = {}


def get_pipeline(model: str):
    """Load (and cache) the text-generation pipeline for a model."""
    if model not in _pipelines:
        _pipelines[model] = pipeline(
            "text-generation",
            model=model,
            dtype='auto',
            token=hf_key,
            device_map='auto',
        )
    return _pipelines[model]


def generate_synthetic_data(model: str, length: int) -> pd.DataFrame:
    length = int(length)

    # Defining the prompt
    SYSTEM_PROMPT = (
        "You create synthetic data for the user, returned as a pandas dataframe. "
        "Respond with ONLY a Python dictionary - no explanation, no markdown fences. "
        "E.g. {'Fruit': ['apple', 'banana', 'pear'], 'calories': [100, 150, 20]}. "
        "Create at least 10 columns and exactly the requested number of rows per column. "
        "Use statistical distributions (normal, uniform, ...) for numeric values "
        "and nominal variables where suitable."
    )
    USER_PROMPT = (
        f"The number of rows requested is {length}. "
        "Create a Python dictionary that I can convert into a dataframe."
    )

    pipe = get_pipeline(model)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ]

    output = pipe(messages, max_new_tokens=2048)
    generated = output[0]['generated_text']
    # Chat models return the full message list; take the last (assistant) message
    text = generated[-1]['content'] if isinstance(generated, list) else generated

    # Extract the dictionary from the raw text (models sometimes add prose or fences)
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"Model did not return a dictionary:\n{text}")
    data = ast.literal_eval(text[start:end + 1])

    return pd.DataFrame(data)

In [ ]:
# Test the generator
df = generate_synthetic_data(MODELS[0], 10)
df

## Gradio frontend

In [ ]:
# Gradio frontend: pick a model and a number of rows, get back a dataframe
test = gr.Interface(
    fn=generate_synthetic_data,
    inputs=[
        gr.Dropdown(choices=MODELS, value=MODELS[0], label="Model"),
        gr.Slider(minimum=1, maximum=1000, value=10, step=1, label="Number of rows"),
    ],
    outputs=gr.Dataframe(label="Synthetic data"),
    title="Synthetic data generator",
    description="Pick an open-source model and a number of rows; get back a generated dataframe.",
)

In [ ]:
test.launch()